In [1]:
import pandas as pd
import zipfile
import os
import numpy as np

In [2]:
path = 'Datasets/AQI_zipfiles'

def read_clean_AQI(path: str) -> pd.DataFrame:
  # unzips and reads in each zipfile in a folder, reads each zipfile into a csv,
  # merges them into one dataframe and cleans up columns
  dict_dataframes = {}
  for root, dirs, files in os.walk(path):
    for file in files:
      file_name = file.split('.')[0]
      zf = zipfile.ZipFile(f'{path}/{file}')
      df = pd.read_csv(zf.open(f'{file_name}.csv'))
      dict_dataframes[file_name] = df


  df = pd.concat(dict_dataframes.values(),ignore_index=True)
  return df

df = read_clean_AQI(path)
df['Percent Moderate Days'] = df['Moderate Days'] / df['Days with AQI']

df.describe()
df.to_csv('Datasets/AQI_merged.csv')

df.head()

,State,County,Year,Days with AQI,Good Days,Moderate Days,Unhealthy for Sensitive Groups Days,Unhealthy Days,Very Unhealthy Days,Hazardous Days,Max AQI,90th Percentile AQI,Median AQI,Days CO,Days NO2,Days Ozone,Days PM2.5,Days PM10,Percent Moderate Days
0,Alabama,Calhoun,1996,53,53,0,0,0,0,0,37,25,17,0,0,0,0,53,0.000000
1,Alabama,Clay,1996,242,157,66,17,2,0,0,161,93,45,0,0,242,0,0,0.272727
2,Alabama,Colbert,1996,54,54,0,0,0,0,0,44,27,14,0,0,0,0,54,0.000000
3,Alabama,DeKalb,1996,225,147,56,16,6,0,0,182,100,41,0,0,207,0,18,0.248889
4,Alabama,Elmore,1996,236,164,62,9,1,0,0,174,84,44,0,0,236,0,0,0.262712


In [3]:
# print list of states to clean up
df.State.unique()

array(['Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Canada',
       'Colorado', 'Connecticut', 'Country Of Mexico', 'Delaware',
       'District Of Columbia', 'Florida', 'Georgia', 'Hawaii', 'Idaho',
       'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana',
       'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota',
       'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada',
       'New Hampshire', 'New Jersey', 'New Mexico', 'New York',
       'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma', 'Oregon',
       'Pennsylvania', 'Puerto Rico', 'Rhode Island', 'South Carolina',
       'South Dakota', 'Tennessee', 'Texas', 'Utah', 'Vermont',
       'Virgin Islands', 'Virginia', 'Washington', 'West Virginia',
       'Wisconsin', 'Wyoming', 'Guam'], dtype=object)

In [4]:
# remove non-US states from state
df = df[~df['State'].isin(['Canada','Guam','Virgin Islands','Country Of Mexico', 'Puerto Rico'])]
print(f'There are {len(df.State.unique())} states in the dataset.')
df.State.unique()

There are 51 states in the dataset.


array(['Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California',
       'Colorado', 'Connecticut', 'Delaware', 'District Of Columbia',
       'Florida', 'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Indiana',
       'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland',
       'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi',
       'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire',
       'New Jersey', 'New Mexico', 'New York', 'North Carolina',
       'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania',
       'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee',
       'Texas', 'Utah', 'Vermont', 'Virginia', 'Washington',
       'West Virginia', 'Wisconsin', 'Wyoming'], dtype=object)

In [5]:
df.sample(10)

,State,County,Year,Days with AQI,Good Days,Moderate Days,Unhealthy for Sensitive Groups Days,Unhealthy Days,Very Unhealthy Days,Hazardous Days,Max AQI,90th Percentile AQI,Median AQI,Days CO,Days NO2,Days Ozone,Days PM2.5,Days PM10,Percent Moderate Days
7556,Michigan,Keweenaw,1991,31,29,2,0,0,0,0,63,38,23,0,0,0,31,0,0.064516
15818,California,Madera,2020,366,104,209,34,15,4,0,275,113,64,0,0,173,174,19,0.571038
35996,Arkansas,Union,2017,365,164,201,0,0,0,0,92,62,52,0,0,0,365,0,0.550685
37632,New York,Oswego,2003,209,155,43,6,3,2,0,204,84,44,0,0,209,0,0,0.205742
40998,Michigan,Washtenaw,1998,347,241,73,28,5,0,0,185,100,40,0,0,347,0,0,0.210375
3740,Indiana,Vermillion,1995,60,56,4,0,0,0,0,55,45,18,0,0,0,0,60,0.066667
12768,Colorado,Delta,2018,297,264,33,0,0,0,0,85,52,36,0,1,199,58,39,0.111111
30491,Wisconsin,Forest,2015,362,318,43,1,0,0,0,102,52,35,0,0,294,68,0,0.118785
15408,Ohio,Warren,2008,258,113,127,16,2,0,0,166,93,52,0,0,171,87,0,0.492248
36317,Louisiana,Lafourche,2017,365,300,65,0,0,0,0,87,54,38,0,0,230,135,0,0.178082


In [6]:
# filter dataframe to only include counties with at least 50 days with AQI that year to remove small sample sizes
#county_df_with_fifty_days = df[df['Days with AQI'] <= 1]
#county_df_with_fifty_days.describe()

In [52]:
# simple aggregation by state with median AQI by county -- I took the median, but perhaps mean would be better

simple_aggregation_median_AQI = df.groupby(['State','Year'])[['Median AQI', 'Days with AQI']]


def wavg(group):
    median = group['Median AQI']
    days = group['Days with AQI']

    return (median * days).sum() / days.sum()
simple_aggregation_median_AQI = pd.DataFrame(simple_aggregation_median_AQI.apply(wavg))
simple_aggregation_median_AQI = simple_aggregation_median_AQI.reset_index()
simple_aggregation_median_AQI = simple_aggregation_median_AQI.rename(columns={0: 'Median AQI'})


In [54]:
simple_aggregation_median_AQI.to_csv('Datasets/AQI_agg.csv')

In [ ]:
# filters to only look at days with AQI and moderate days columns
# these and/or other filters should be moved into a function once we finalize which column we want to use as our target
df = df[['State','County','Year','Days with AQI','Moderate Days']]

# calculates the percentage moderate days
df['Percent Moderate Days'] = df['Moderate Days'] / df['Days with AQI']
df.describe()

In [ ]:
# filter dataframe to only include counties with at least 50 days with AQI that year to remove small sample sizes
county_df_with_fifty_days = df[df['Days with AQI'] >= 50]
county_df_with_fifty_days.describe()

In [ ]:
# simple aggregation by state of mean percent moderate days
simple_aggregation_percent_moderate = county_df_with_fifty_days.groupby(['State','Year'])['Percent Moderate Days'].mean().reset_index()
simple_aggregation_percent_moderate.describe()

In [ ]:
simple_aggregation_percent_moderate.sample(10)

In [ ]:
df[df['State'] == 'South Dakota'].Year.unique()

In [ ]:
df['Days with AQI'].unique()